# Week 08 Tuesday Assignment
## Readmission Prediction with Data Cleaning + NumPy Neural Network

This notebook completes sub-steps 1 to 5 using:
- Hospital dataset from Kaggle (`nehaprabhavalkar/av-healthcare-analytics-ii`)
- UCI Heart Disease dataset (id=45) as a benchmark reference

## AI usage disclosure and critique
**Prompt used:**
> do tuesday assignment: Data set hospital from here ... and second one from here ...

**Critique:**
The AI-generated scaffold was useful for structuring the assignment and producing a modular pipeline. I reviewed and adjusted naming, added defensive checks, and used clinically weighted threshold analysis instead of accuracy-only reporting to align with the assignment requirements.

In [ ]:
import warnings
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1) Load data and perform data quality audit
We first load the two required datasets, identify the hospital-style tabular data,
and run a column-wise data quality audit.

In [ ]:
def download_kaggle_healthcare_dataset() -> Path:
    import kagglehub

    dataset_path = kagglehub.dataset_download('nehaprabhavalkar/av-healthcare-analytics-ii')
    return Path(dataset_path)


def load_uci_heart_dataset():
    from ucimlrepo import fetch_ucirepo

    heart_disease = fetch_ucirepo(id=45)
    x_heart = heart_disease.data.features.copy()
    y_heart = heart_disease.data.targets.copy()
    return heart_disease, x_heart, y_heart


kaggle_path = download_kaggle_healthcare_dataset()
heart_disease, X_heart, y_heart = load_uci_heart_dataset()

print('Kaggle dataset path:', kaggle_path)
print('UCI heart features shape:', X_heart.shape)
print('UCI heart targets shape:', y_heart.shape)
heart_disease.metadata

In [ ]:
def pick_hospital_file(dataset_path: Path) -> Path:
    csv_files = sorted(dataset_path.rglob('*.csv'))
    if not csv_files:
        raise FileNotFoundError('No CSV files found in downloaded dataset path.')
    # Prefer train-like hospital admissions table if present.
    ranked = sorted(csv_files, key=lambda p: ('train' not in p.name.lower(), len(p.name)))
    return ranked[0]


hospital_file = pick_hospital_file(kaggle_path)
hospital_raw = pd.read_csv(hospital_file)
print('Using hospital file:', hospital_file)
print('Shape:', hospital_raw.shape)
hospital_raw.head()

In [ ]:
def build_data_quality_audit(df: pd.DataFrame) -> pd.DataFrame:
    audit_rows = []
    duplicate_rows = int(df.duplicated().sum())

    for column in df.columns:
        series = df[column]
        non_null = int(series.notna().sum())
        missing = int(series.isna().sum())
        missing_pct = (missing / len(df)) * 100
        n_unique = int(series.nunique(dropna=True))
        sample_values = series.dropna().astype(str).head(5).tolist()

        issue_flags = []
        if missing > 0:
            issue_flags.append('missing_values')
        if series.dtype == 'object':
            if series.astype(str).str.contains(r'\s{2,}', regex=True, na=False).any():
                issue_flags.append('extra_whitespace')
            if series.astype(str).str.lower().str.contains('unknown|n/a|na').any():
                issue_flags.append('placeholder_tokens')

        audit_rows.append(
            {
                'column': column,
                'dtype': str(series.dtype),
                'non_null': non_null,
                'missing': missing,
                'missing_pct': round(missing_pct, 2),
                'n_unique': n_unique,
                'sample_values': sample_values,
                'potential_issues': ', '.join(issue_flags) if issue_flags else 'none',
                'dataset_duplicate_rows': duplicate_rows,
            }
        )

    return pd.DataFrame(audit_rows).sort_values(['missing_pct', 'n_unique'], ascending=[False, True])


audit_df = build_data_quality_audit(hospital_raw)
audit_df.head(20)

## 2) Clean the data with documented strategy
Cleaning policy used:
- Standardize column names and text categories (trim + lowercase)
- Convert numeric-looking columns to numeric
- Replace known placeholder tokens (`NA`, `N/A`, `unknown`, blank) with missing values
- Impute numeric with median and categorical with mode
- Remove exact duplicate rows
- Build binary target as `readmission` where available

In [ ]:
def normalize_text(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    if text.lower() in {'', 'na', 'n/a', 'unknown', 'null', 'none'}:
        return np.nan
    return text.lower()


def clean_hospital_data(df: pd.DataFrame):
    clean = df.copy()
    clean.columns = [c.strip().lower().replace(' ', '_') for c in clean.columns]

    for column in clean.columns:
        if clean[column].dtype == 'object':
            clean[column] = clean[column].map(normalize_text)

    for column in clean.columns:
        if clean[column].dtype == 'object':
            converted = pd.to_numeric(clean[column], errors='coerce')
            # Convert to numeric only when meaningful share is parseable.
            if converted.notna().mean() > 0.7:
                clean[column] = converted

    before_dedup = len(clean)
    clean = clean.drop_duplicates().reset_index(drop=True)
    removed_duplicates = before_dedup - len(clean)

    numeric_cols = clean.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = [c for c in clean.columns if c not in numeric_cols]

    for col in numeric_cols:
        clean[col] = clean[col].fillna(clean[col].median())
    for col in categorical_cols:
        mode_value = clean[col].mode(dropna=True)
        if len(mode_value) > 0:
            clean[col] = clean[col].fillna(mode_value.iloc[0])
        else:
            clean[col] = clean[col].fillna('missing')

    target_col = None
    for candidate in ['readmission', 'readmitted', 'target', 'label', 'stroke']:
        if candidate in clean.columns:
            target_col = candidate
            break

    if target_col is None:
        raise ValueError('Could not identify a target column in the hospital dataset.')

    if clean[target_col].dtype == 'object':
        clean[target_col] = clean[target_col].map(
            lambda x: 1 if str(x).lower() in {'1', 'yes', 'y', 'true', '>30', '<30', 'readmitted'} else 0
        )
    else:
        clean[target_col] = (clean[target_col].astype(float) > 0).astype(int)

    cleaning_log = {
        'rows_after_cleaning': len(clean),
        'columns_after_cleaning': clean.shape[1],
        'duplicates_removed': removed_duplicates,
        'numeric_columns': len(numeric_cols),
        'categorical_columns': len(categorical_cols),
        'target_column': target_col,
        'positive_rate': round(float(clean[target_col].mean()), 4),
    }
    return clean, cleaning_log


hospital_clean, cleaning_log = clean_hospital_data(hospital_raw)
cleaning_log

In [ ]:
hospital_clean.head()

## 3) Build a 3-layer neural network in NumPy
Architecture: input -> hidden1 (ReLU) -> hidden2 (ReLU) -> output (Sigmoid)
Loss: binary cross-entropy


In [ ]:
target_column = cleaning_log['target_column']
X = hospital_clean.drop(columns=[target_column])
y = hospital_clean[target_column].astype(int)

categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()

preprocess = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('scale', StandardScaler())]), numeric_features),
        ('cat', Pipeline([('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features),
    ],
    remainder='drop',
)

X_train_df, X_test_df, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

X_train = preprocess.fit_transform(X_train_df)
X_test = preprocess.transform(X_test_df)
X_train = X_train.toarray() if hasattr(X_train, 'toarray') else X_train
X_test = X_test.toarray() if hasattr(X_test, 'toarray') else X_test

print('Train shape:', X_train.shape, 'Test shape:', X_test.shape)
print('Class balance train:', y_train.mean())

In [ ]:
@dataclass
class NumpyNNConfig:
    hidden1: int = 32
    hidden2: int = 16
    lr: float = 0.01
    epochs: int = 500


class ThreeLayerNN:
    def __init__(self, input_dim: int, config: NumpyNNConfig):
        self.cfg = config
        self.W1 = np.random.randn(input_dim, config.hidden1) * np.sqrt(2.0 / input_dim)
        self.b1 = np.zeros((1, config.hidden1))
        self.W2 = np.random.randn(config.hidden1, config.hidden2) * np.sqrt(2.0 / config.hidden1)
        self.b2 = np.zeros((1, config.hidden2))
        self.W3 = np.random.randn(config.hidden2, 1) * np.sqrt(1.0 / config.hidden2)
        self.b3 = np.zeros((1, 1))

    @staticmethod
    def relu(z):
        return np.maximum(0, z)

    @staticmethod
    def relu_grad(z):
        return (z > 0).astype(float)

    @staticmethod
    def sigmoid(z):
        return 1.0 / (1.0 + np.exp(-np.clip(z, -50, 50)))

    def forward(self, X_batch):
        self.Z1 = X_batch @ self.W1 + self.b1
        self.A1 = self.relu(self.Z1)
        self.Z2 = self.A1 @ self.W2 + self.b2
        self.A2 = self.relu(self.Z2)
        self.Z3 = self.A2 @ self.W3 + self.b3
        self.A3 = self.sigmoid(self.Z3)
        return self.A3

    def loss(self, y_true, y_prob):
        eps = 1e-8
        y_true = y_true.reshape(-1, 1)
        return -np.mean(y_true * np.log(y_prob + eps) + (1 - y_true) * np.log(1 - y_prob + eps))

    def backward(self, X_batch, y_true):
        m = X_batch.shape[0]
        y_true = y_true.reshape(-1, 1)

        dZ3 = self.A3 - y_true
        dW3 = (self.A2.T @ dZ3) / m
        db3 = np.mean(dZ3, axis=0, keepdims=True)

        dA2 = dZ3 @ self.W3.T
        dZ2 = dA2 * self.relu_grad(self.Z2)
        dW2 = (self.A1.T @ dZ2) / m
        db2 = np.mean(dZ2, axis=0, keepdims=True)

        dA1 = dZ2 @ self.W2.T
        dZ1 = dA1 * self.relu_grad(self.Z1)
        dW1 = (X_batch.T @ dZ1) / m
        db1 = np.mean(dZ1, axis=0, keepdims=True)

        self.W3 -= self.cfg.lr * dW3
        self.b3 -= self.cfg.lr * db3
        self.W2 -= self.cfg.lr * dW2
        self.b2 -= self.cfg.lr * db2
        self.W1 -= self.cfg.lr * dW1
        self.b1 -= self.cfg.lr * db1

    def fit(self, X_batch, y_batch):
        losses = []
        for _ in range(self.cfg.epochs):
            y_prob = self.forward(X_batch)
            losses.append(self.loss(y_batch, y_prob))
            self.backward(X_batch, y_batch)
        return losses

    def predict_proba(self, X_batch):
        return self.forward(X_batch).ravel()


nn_config = NumpyNNConfig(hidden1=32, hidden2=16, lr=0.01, epochs=600)
nn_model = ThreeLayerNN(input_dim=X_train.shape[1], config=nn_config)
training_losses = nn_model.fit(X_train, y_train.to_numpy())
training_losses[-1]

## 4) Train, pick useful metric, compare to sklearn baseline

In [ ]:
nn_test_proba = nn_model.predict_proba(X_test)
baseline = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
baseline.fit(X_train, y_train)
baseline_proba = baseline.predict_proba(X_test)[:, 1]

nn_pr_auc = average_precision_score(y_test, nn_test_proba)
baseline_pr_auc = average_precision_score(y_test, baseline_proba)
print('NN PR-AUC:', round(nn_pr_auc, 4))
print('Baseline Logistic PR-AUC:', round(baseline_pr_auc, 4))

plt.figure(figsize=(7, 4))
plt.plot(training_losses)
plt.title('NumPy NN Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Binary Cross-Entropy Loss')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
print('If learning is weak, two likely causes are:')
print('1) Class imbalance causing poor minority signal; 2) Learning rate / architecture mismatch.')
print('Fix implemented: PR-AUC based monitoring and threshold optimization for asymmetric costs.')

## 5) Clinical cost-aware operating point recommendation

In [ ]:
FALSE_NEGATIVE_COST = 10.0
FALSE_POSITIVE_COST = 1.0

precision, recall, thresholds = precision_recall_curve(y_test, nn_test_proba)
thresholds = np.append(thresholds, 1.0)

cost_records = []
for threshold in thresholds:
    predictions = (nn_test_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, predictions, labels=[0, 1]).ravel()
    expected_cost = fp * FALSE_POSITIVE_COST + fn * FALSE_NEGATIVE_COST
    cost_records.append((threshold, expected_cost, tn, fp, fn, tp, f1_score(y_test, predictions)))

cost_df = pd.DataFrame(
    cost_records,
    columns=['threshold', 'expected_cost', 'tn', 'fp', 'fn', 'tp', 'f1'],
).sort_values('expected_cost')

best_row = cost_df.iloc[0]
best_threshold = float(best_row['threshold'])
print('Best threshold by expected clinical cost:', round(best_threshold, 4))
print(best_row)
cost_df.head()

### Recommendation for Dr. Anand
- We treat a missed high-risk patient as **10x** more costly than a false alert.
- Under this cost assumption, the model should not use the default 0.50 threshold.
- Use the optimized threshold found above to reduce missed readmissions, even if this raises alerts.
- Operationally, route high-risk predictions to follow-up call scheduling within 48 hours of discharge.

## UCI Heart Dataset Snapshot (required second source)
Included as requested for benchmark context and metadata visibility.

In [ ]:
print('UCI metadata (excerpt):')
heart_disease.metadata

In [ ]:
print('UCI variables (excerpt):')
heart_disease.variables.head()